# SGD with momentum as a torch.optim.Optimizer

Built-in optimizers are ordinary Python classes. Write your own subclass of `torch.optim.Optimizer` that reproduces `torch.optim.SGD` with momentum, dampening, weight decay and Nesterov, then use it to train a linear model.

## The optimizer
Write `class MomentumSGD(torch.optim.Optimizer)` with defaults `lr, momentum=0, dampening=0, weight_decay=0, nesterov=False`. Its `step()` updates every parameter `p` that has a gradient, using its **own group's** hyper-parameters:

1. $g = \nabla p + \lambda\, p$ (weight decay $\lambda$ is added to the gradient, before momentum).
2. If momentum $\mu \ne 0$: on the parameter's first step the buffer is $b = g$ (a copy, **no dampening**); afterwards $b \leftarrow \mu\, b + (1 - \tau)\, g$ with dampening $\tau$. Then the direction is $d = g + \mu\, b$ if Nesterov, else $d = b$.
3. If $\mu = 0$: $d = g$ and no buffer is kept.
4. $p \leftarrow p - \eta\, d$.

Keep the buffer in `self.state[p]`. `step()` must not be recorded by autograd.

## The training run
Write `train_with_custom_sgd(W0, b0, X, Y, steps, lr, momentum, dampening, weight_decay, nesterov)`:

- `X`: float64 `(N, D)`, `Y`: float64 `(N, K)`; `W0`: `(K, D)`, `b0`: `(K,)` initial weights. Copy them; don't modify the inputs.
- Model: $\hat Y = X W^\top + b$. Loss: mean squared error over all $N K$ entries, on the full batch every step.
- Two parameter groups: `[W]` with `weight_decay=weight_decay`, and `[b]` with `weight_decay=0` (biases are not decayed). Both share `lr`, `momentum`, `dampening`, `nesterov`.
- Run `steps` iterations of: zero the gradients, loss, backward, `step()`.

Return a tuple `(losses, W, b, buf_W, buf_b)`:

- `losses`: float64 `(steps,)`, the loss at each step **before** its update.
- `W`, `b`: final parameters.
- `buf_W`, `buf_b`: final momentum buffers, or zero tensors of the parameter's shape when `momentum == 0`.

Nesterov is only used with `momentum > 0` and `dampening == 0`.

This problem runs in **Colab** (PyTorch isn't available in the browser). The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import torch

class MomentumSGD(torch.optim.Optimizer):
    def __init__(self, params, lr, momentum=0.0, dampening=0.0, weight_decay=0.0, nesterov=False):
        defaults = dict(lr=lr, momentum=momentum, dampening=dampening,
                        weight_decay=weight_decay, nesterov=nesterov)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self):
        for group in self.param_groups:
            for p in group["params"]:
                if p.grad is None:
                    continue
                p -= group["lr"] * p.grad          # TODO: weight decay, momentum buffer, nesterov


def train_with_custom_sgd(W0, b0, X, Y, steps, lr, momentum, dampening, weight_decay, nesterov):
    """Return (losses (steps,), W, b, buf_W, buf_b)."""
    raise NotImplementedError


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[0.0]], "dtype": "float64"}, {"__nd__": [0.0], "dtype": "float64"}, {"__nd__": [[1.0], [2.0]], "dtype": "float64"}, {"__nd__": [[2.0], [4.0]], "dtype": "float64"}, 3, 0.1, 0.9, 0.0, 0.0, false], "e": {"__tup__": [{"__nd__": [10.0, 1.0599999999999998, 2.7346000000000013], "dtype": "float64"}, {"__nd__": [[2.8120000000000003]], "dtype": "float64"}, {"__nd__": [1.6380000000000001], "dtype": "float64"}, {"__nd__": [[-5.919999999999999]], "dtype": "float64"}, {"__nd__": [-3.1799999999999997], "dtype": "float64"}]}}, {"a": [{"__nd__": [[1.0, -1.0]], "dtype": "float64"}, {"__nd__": [0.5], "dtype": "float64"}, {"__nd__": [[1.0, 0.0], [0.0, 1.0]], "dtype": "float64"}, {"__nd__": [[1.0], [0.0]], "dtype": "float64"}, 4, 0.2, 0.5, 0.0, 0.1, true], "e": {"__tup__": [{"__nd__": [0.25, 0.10240000000000005, 0.028696359999999997, 0.004019306403999987], "dtype": "float64"}, {"__nd__": [[0.49810966, -0.49810965999999995]], "dtype": "float64"}, {"__nd__": [0.5], "dtype": "float64"}, {"__nd__": [[0.4134077999999998, -0.41340779999999994]], "dtype": "float64"}, {"__nd__": [-1.1102230246251565e-16], "dtype": "float64"}]}}, {"a": [{"__nd__": [[0.14055981660180925, 0.07474079874793504, 0.019381564626462], [0.11116332052239922, -0.02055230499057925, -0.09258995736483681]], "dtype": "float64"}, {"__nd__": [0.0, 0.0], "dtype": "float64"}, {"__nd__": [[2.0409191213851825, -2.5556650313141818, 0.41809884672577885], [-0.5677696061279298, -0.45264929211044586, -0.2155971630897659], [-2.019986129147251, -0.23193237764418947, -0.8652130762749417], [3.3229995166448827, 0.22578661322792176, -0.3526307943415954], [-0.2812874181513504, -0.6680463461089501, -1.0551505512051214], [-0.39080097723465473, 0.48194538850678587, -0.2385536065733667], [0.9577587029597641, -0.19980212906658, 0.024259565076664623], [1.545820851212812, 0.5451055226876446, -0.505228735614018]], "dtype": "float64"}, {"__nd__": [[-5.009073977801006, 2.6821214568137743], [-0.1313413592130555, 0.15708874180899662], [0.6404044481632923, -1.329523983887684], [0.13241320909301618, 1.9839721313404606], [-0.5802715889765584, -0.6963777773717952], [1.5898067815966466, -0.010231059086709071], [-0.1121357014988178, 0.9882384725764702], [1.3978562017588732, 0.6769111255226594]], "dtype": "float64"}, 6, 0.05, 0.0, 0.0, 0.0, false], "e": {"__tup__": [{"__nd__": [2.6817851831368036, 2.2202782337903986, 1.8690194128996886, 1.5978852604767377, 1.3853524233532422, 1.2160099979345318], "dtype": "float64"}, {"__nd__": [[-0.16169465271605707, 0.5370969077618452, -0.0920415721757607], [0.47399027116838793, -0.16463661573072427, -0.023407804136165094]], "dtype": "float64"}, {"__nd__": [-0.04044032613115998, 0.08696189248497778], "dtype": "float64"}, {"__nd__": [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], "dtype": "float64"}, {"__nd__": [0.0, 0.0], "dtype": "float64"}]}}, {"a": [{"__nd__": [[-0.1730013451272522, -0.0004414232621967487, 0.12135638252860817]], "dtype": "float64"}, {"__nd__": [0.0], "dtype": "float64"}, {"__nd__": [[0.584058311025248, 0.5825384186556901, -0.2148289111268558], [-0.7828085779639662, 0.22915390521326254, -2.4938942784579905], [0.690124770162812, 0.4913682607449912, -1.6388571438904884], [0.06135350983817159, -0.9640996635412404, 0.7572210447581504], [-2.034167273443428, -0.9144945379945887, 0.7095799877420675], [1.156401048432157, -2.158005380126208, -0.49803984475130336], [0.32802009254257697, -0.6092161379498706, 1.5906402313231438], [-1.1912266816177808, 0.354531946286926, -1.0484055185445111], [1.4059629431348852, -0.021651229055558368, -0.3722505640006159], [-1.7181849497326165, 1.6818255450666806, 0.7527785926973876]], "dtype": "float64"}, {"__nd__": [[1.4640361448009622], [-0.7801045896652753], [0.9268193349634495], [-0.1493482040771899], [-1.9716950219745004], [-1.3176975688614703], [0.5773379067943184], [-0.33791926136758954], [1.462381451230303], [1.2569363153316417]], "dtype": "float64"}, 8, 0.05, 0.9, 0.0, 0.0, false], "e": {"__tup__": [{"__nd__": [1.562903594681897, 1.3520384788570765, 1.0153821269573124, 0.6585576410088192, 0.3673050307972662, 0.1845635232463418, 0.10968824980913054, 0.11375218734447401], "dtype": "float64"}, {"__nd__": [[1.0551691887861645, 1.38365169468174, 0.26311590929205964]], "dtype": "float64"}, {"__nd__": [0.39738544641974866], "dtype": "float64"}, {"__nd__": [[-2.257564758758068, -3.419696184021645, -1.5521780525193083]], "dtype": "float64"}, {"__nd__": [-1.6706555189143382], "dtype": "float64"}]}}, {"a": [{"__nd__": [[-0.055137240798046376, 0.2160906258701247, 0.10192065653546221, 0.2175575322620394], [-0.0026589187155779802, -0.03830885502613072, 0.01670480021669256, 0.07345765026061907]], "dtype": "float64"}, {"__nd__": [0.0, 0.0], "dtype": "float64"}, {"__nd__": [[0.7570580592965243, 0.21565078369996044, -0.3171556440173552, 0.2932336958002246], [-0.24333508574217566, 0.817206580492595, -0.7944473388868819, 0.13423994708633882], [-0.11078013611159404, 0.5433593895301524, 0.22463852364937692, 2.550034636307906], [1.498654758135483, 1.4967371655185107, -2.0395038375946424, -0.3403166247023773], [-0.6086106159129299, 0.5327215998890392, -2.279026489055327, 1.1744986790091876], [1.0669833108953142, -1.3020708582457947, -0.9785485286214127, -0.8011720107811816], [0.043295900283144985, 0.6409710646894711, 2.0478860553573326, -0.19744542988767094], [0.7675025589036195, 0.15541781005943467, 1.7599262839082537, 0.7421578612115688], [1.3685504508744795, -1.0776751897834627, -0.19224071152595104, -0.8137724218420405], [1.5049474040083073, 0.6576399038177728, -0.3051444257775161, -0.4524678870704014], [0.4846648782067015, -0.7014955301795535, -0.9305888762121555, 0.4812744922715323], [2.463132032105226, -0.24613355406481327, -0.5558657807935545, -1.171156834025543]], "dtype": "float64"}, {"__nd__": [[-0.43783637010769877, 0.42628439907054616], [1.1962190748729338, 0.03514670050002927], [1.45655653306689, -3.45224132662679], [-0.8674577309837168, 1.542807725950953], [1.1552563716270925, -1.7898997036770208], [-2.40429135143676, 2.3270088895494996], [1.4321920805257269, 1.1530133197143313], [0.24759021723491986, -0.010825234019325836], [-2.3369153637858733, 2.53656625186418], [-1.0098847555414463, 1.9667831520458838], [-0.825661654330637, -0.12266461372480258], [-3.359071800487623, 3.5937174789838515]], "dtype": "float64"}, 7, 0.03, 0.8, 0.3, 0.01, false], "e": {"__tup__": [{"__nd__": [2.9787971698574918, 2.6761449618592636, 2.267440207854286, 1.824427930976424, 1.402507108394026, 1.0373136086648602, 0.7458150305252417], "dtype": "float64"}, {"__nd__": [[-0.524772667103623, 0.4135216206308347, 0.2556074084478669, 0.49987047870142814], [0.6559696719679049, -0.18440923445486387, -0.05297786202459158, -0.5966685253973005]], "dtype": "float64"}, {"__nd__": [-0.14534593336923252, 0.22694386628804006], "dtype": "float64"}, {"__nd__": [[2.416030601740518, -1.1456228512997442, -0.6835895513742061, -1.4534348667805344], [-3.3221425235528868, 0.7575043705993835, 0.15073733461461394, 3.6659839556390157]], "dtype": "float64"}, {"__nd__": [0.6071784902770919, -1.0332973041470463], "dtype": "float64"}]}}, {"a": [{"__nd__": [[0.07354570881988916, -0.11458272768549103], [-0.06588555689831765, -0.008033731382935666], [-0.056592532701236646, 0.1743754609849686]], "dtype": "float64"}, {"__nd__": [0.0, 0.0, 0.0], "dtype": "float64"}, {"__nd__": [[-0.5874276868740115, 0.3797017469724474], [-0.01680427892083426, 1.6156817605764437], [-0.6627018123008657, 1.046161771728719], [-0.6438635355400087, -0.9606341931074224], [-0.7102994273756232, -1.1901858618422472], [0.1463503600664521, 1.0312607960786646], [0.1643148785347983, 0.6243251121848756], [1.6321741955751323, 0.27002644717885166], [0.19516236096309458, -0.27506146796027464]], "dtype": "float64"}, {"__nd__": [[1.685771402468797, -0.03404915566658179, 1.995425409838749], [1.5255191521821796, 0.44600034736500943, 2.3542233681918034], [2.235135242774188, -0.03734198232506841, 2.759560701978722], [0.8895918833972192, -0.18924565130926155, 0.5327628345144332], [0.7457217085899724, 0.08942810190122151, 0.5335261727481141], [0.742610205829352, 0.6109117444428732, 1.3923105248002223], [0.5385525906415161, 0.6690611793593353, 0.8686964161167469], [-2.1254515681227844, 1.709491745382555, -2.0774139380841046], [0.017248601785846952, 0.5290881163543437, -0.09116103513514726]], "dtype": "float64"}, 6, 0.04, 0.9, 0.0, 0.05, true], "e": {"__tup__": [{"__nd__": [1.673346231967153, 1.5518894639898615, 1.394101696203748, 1.2169908758784744, 1.0356970008753343, 0.862332495550462], "dtype": "float64"}, {"__nd__": [[-0.35652234768603397, 0.16872887155305574], [0.1459277811651048, 0.10728440478326041], [-0.4648655840535127, 0.5241052919654104]], "dtype": "float64"}, {"__nd__": [0.3559186257319141, 0.20982087806317493, 0.4218298969766024], "dtype": "float64"}, {"__nd__": [[2.191575147858597, -1.3733825452193045], [-1.0740039840954951, -0.5298082606616625], [2.0830659944478493, -1.6907709134590116]], "dtype": "float64"}, {"__nd__": [-1.6970516450894029, -1.0172041817747979, -2.011880896063638], "dtype": "float64"}]}}, {"a": [{"__nd__": [[-0.008157358819027507, 0.09946628839764439, 0.004389702264947434, -0.22414815436236601, -0.05726513109844991], [0.018985359658138634, -0.11509497089580939, -0.13474107332725924, 0.03661858537229255, -0.0649414999777468]], "dtype": "float64"}, {"__nd__": [0.0, 0.0], "dtype": "float64"}, {"__nd__": [[0.20837211615824647, -1.010582036354199, -0.7881089276247821, -0.05747025825572099, 2.2957788082586084], [-0.17826432182306642, 0.12747641923980615, 0.5140453781581755, -0.0401259362630417, 2.280660872566708], [-0.5315249470251602, 0.7442031251457859, 0.16043796898357113, 0.6211963732164166, -1.1766429584197382], [1.768112695107191, -0.12152750716346146, 0.03416326206751611, -0.9349762618687502, -0.7296952681832494], [0.5642394331680308, 0.9864998861106795, 0.7532546907537343, 1.2089702496885697, 0.7144889121895972], [0.02846183335685169, 0.8365259894445134, 0.5936032404079788, -0.10057526654145102, 0.726072718540168]], "dtype": "float64"}, {"__nd__": [[2.4795692317313955, -4.102228485468043], [3.9141091422453655, -3.5940907509879105], [-1.5863875739616782, 2.8984977410346704], [2.095772729425787, 2.134885965255389], [3.011448132937679, -0.11211172894427196], [2.1747963004999082, -0.15285738036763807]], "dtype": "float64"}, 1, 0.1, 0.9, 0.5, 0.1, false], "e": {"__tup__": [{"__nd__": [7.160434927670949], "dtype": "float64"}, {"__nd__": [[0.08996107653406209, 0.12035813747826586, 0.0614341373175049, -0.21135663926178316, 0.271977163561587], [0.05060463996379518, -0.016169971963311085, -0.09638785685602227, 0.04208014885249673, -0.43221028938912714]], "dtype": "float64"}, {"__nd__": [0.20558532981329125, -0.03951933339978643], "dtype": "float64"}, {"__nd__": [[-0.981184353530896, -0.20891849080621472, -0.5704443505255746, -0.1279151510058286, -3.292422946600369], [-0.3161928030565655, -0.989249989324983, -0.3835321647123696, -0.054615634802041756, 3.6726878941138033]], "dtype": "float64"}, {"__nd__": [-2.0558532981329125, 0.39519333399786427], "dtype": "float64"}]}}, {"a": [{"__nd__": [[0.04619216464173159, -0.07245215730251725, 0.1429494960235201, 0.14285101494980432], [0.09807761712193035, 0.07888168682681605, 0.11556517800640663, 0.054325904634565825]], "dtype": "float64"}, {"__nd__": [0.0, 0.0], "dtype": "float64"}, {"__nd__": [[-1.6958272209806426, -0.6841151736064445, 0.8607375733597276, -0.47374607008243474], [0.9347725714518614, 1.5272727626498412, 0.03251540049551873, -1.1296833434653808], [-1.3015650956398057, -0.026253854187163014, 0.8810556770082387, 0.2690153283133568], [0.5472571901882256, -0.4433417809053284, 0.3004227591479358, -0.4713087658775225], [-0.32560990354708425, -1.292246473319263, -1.5722129972246757, -0.4819566918389441], [-1.1075993405277864, -0.9587457201198237, 0.6710474629653865, -0.10708092147336722], [2.914125888022547, 0.926342075423221, -0.6331876151795489, 0.7780073420949039], [0.3568038879399853, -0.6856810795173335, 1.102594967084939, -0.7753286078642612], [-3.0583812523616074, 0.7940647591398647, -0.6172591186177742, 1.4860374923545419], [-0.6496722067169012, -1.1813542467853593, 1.548461665941953, -1.0754983251353518], [0.20339369150255643, 1.4497271957967346, 0.11449734341089023, -0.08058491531778637], [0.10242343723857332, 0.8390536522388322, -0.8068121596797956, 0.6106409992957679], [0.5770754169394714, 2.332712025382542, -0.3365255299285121, -0.9148298592004128], [0.735687023006662, -0.3971572496742207, 0.28911616638337856, 0.13257652253025445], [1.1683939613256054, -0.6568374851700608, -0.09227952885816246, -2.5521126492129294]], "dtype": "float64"}, {"__nd__": [[1.409832336793618, -0.5381049843024084], [-1.5381694014384122, 2.1602702594287666], [0.825072256704581, -1.24866082674516], [-0.14644549586215316, 2.3334591867565906], [3.4750467281016033, 0.6525084300679964], [1.2760508989001023, 0.09489155583063114], [-1.7575025365701697, 3.171391707544528], [-0.7497491596258217, 3.0824013158138013], [3.506478198418496, -6.84750743531709], [-0.11502093758752788, 2.5211897098888727], [-0.7671884972042999, -0.08959735646253096], [0.8902741053214218, -1.2180254092762899], [-1.2202037062722475, 0.6567315784414631], [-0.2619481748104378, 1.7859646655872383], [-0.44955884425467696, 5.875579468444038]], "dtype": "float64"}, 10, 0.02, 0.0, 0.0, 0.2, false], "e": {"__tup__": [{"__nd__": [5.283137839932271, 4.941897347769534, 4.625989653997847, 4.333409287822183, 4.062315806752971, 3.811019860789928, 3.5779704552899663, 3.361743308008772, 3.161030205010106, 2.9746292685245637], "dtype": "float64"}, {"__nd__": [[-0.22593936629634903, -0.16845331195531654, 0.05220125407606818, 0.20049740861712112], [0.546534522502524, -0.06754586075943664, 0.2089128816760476, -0.31880109892605746]], "dtype": "float64"}, {"__nd__": [0.0628996356937939, 0.1422263791012061], "dtype": "float64"}, {"__nd__": [[0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0]], "dtype": "float64"}, {"__nd__": [0.0, 0.0], "dtype": "float64"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(train_with_custom_sgd, json.loads(_TESTS), 'torch', *[1e-06, 1e-08])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")